# Preprocessing for modeling\n\nStarts from `model_cleaned_listings.csv` (pre-one-hot). Encodes `Locality`/`Society` (too high-cardinality for one-hot), checks multicollinearity (VIF), splits train/test **before** target encoding to avoid leakage, and scales numeric features.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

RAW = "../raw"
df = pd.read_csv(f"{RAW}/model_cleaned_listings.csv", low_memory=False)
df.shape

(32587, 34)

## Train/test split (before any encoding that looks at the target)

In [2]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
len(train_df), len(test_df)

(26069, 6518)

## Target-encode Locality &amp; Society\n\nMean `log_price` per category, smoothed toward the global mean by category frequency so rare localities don't get an overconfident estimate. Fit on train only, applied to both splits — unseen test categories fall back to the global mean.

In [3]:
GLOBAL_MEAN = train_df["log_price"].mean()
SMOOTHING = 20

def fit_target_encoding(train, col, target="log_price"):
    stats = train.groupby(col)[target].agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + GLOBAL_MEAN * SMOOTHING) / (stats["count"] + SMOOTHING)
    return smoothed

def apply_target_encoding(df, col, mapping, out_col):
    df[out_col] = df[col].map(mapping).fillna(GLOBAL_MEAN)
    return df

for col in ["Locality", "Society"]:
    mapping = fit_target_encoding(train_df, col)
    train_df = apply_target_encoding(train_df, col, mapping, f"{col}_target_enc")
    test_df = apply_target_encoding(test_df, col, mapping, f"{col}_target_enc")

train_df[["Locality","Locality_target_enc","Society","Society_target_enc"]].head()

,Locality,Locality_target_enc,Society,Society_target_enc
2862,"SBH Colony, LB Nagar, Hyderabad",16.306694,"in SBH Colony, LB Nagar, Hyderabad",16.306694
16147,Meerpet,15.837289,Vijaya Residency Ma...,16.238421
14177,Gopanpalle,16.499754,Apartment,15.781529
24391,Mallampet,16.099903,Hi Rise Pvr Meadows,16.315449
26598,Banjara Hills,16.518881,Samskruthi Palace,16.290252


## Frequency-encode Locality &amp; Society (a second, leakage-free signal)

In [4]:
for col in ["Locality", "Society"]:
    freq = train_df[col].value_counts()
    train_df[f"{col}_freq"] = train_df[col].map(freq).fillna(0)
    test_df[f"{col}_freq"] = test_df[col].map(freq).fillna(0)

## Multicollinearity check (VIF)

In [5]:
vif_cols = ["BHK","Area_Sqft","Bathrooms","Balconies","Car_Parking","Floor_Number",
            "Total_Floors","Property_Age_Years","floor_ratio","Locality_target_enc",
            "Society_target_enc","Locality_freq","Society_freq"]
X_vif = train_df[vif_cols].fillna(0)
vif = pd.DataFrame({
    "feature": vif_cols,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(len(vif_cols))],
}).sort_values("VIF", ascending=False)
vif

,feature,VIF
1,Area_Sqft,2.409424
0,BHK,2.277221
2,Bathrooms,2.198634
10,Society_target_enc,1.829185
12,Society_freq,1.604750
9,Locality_target_enc,1.533940
3,Balconies,1.080857
8,floor_ratio,1.060606
7,Property_Age_Years,1.047247
11,Locality_freq,1.040838


VIF &gt; 10 flags a feature that's largely explained by the others — expect `Bathrooms`/`BHK`/`Area_Sqft` to run high (they move together by construction) and `Locality_target_enc`/`Locality_freq` to be correlated with each other. Drop or combine flagged pairs rather than feeding both into a linear model; tree-based models tolerate this collinearity fine.

## Scale numeric features (fit on train only)

In [6]:
numeric_cols = ["BHK","Area_Sqft","Bathrooms","Balconies","Car_Parking","Floor_Number",
                "Total_Floors","Property_Age_Years","floor_ratio",
                "Locality_target_enc","Society_target_enc","Locality_freq","Society_freq",
                "Cement_Price_Rs_per_bag_50kg","Steel_TMT_Price_Rs_per_tonne"]

scaler = StandardScaler()
train_scaled = pd.DataFrame(scaler.fit_transform(train_df[numeric_cols].fillna(0)),
                             columns=[f"{c}_scaled" for c in numeric_cols], index=train_df.index)
test_scaled = pd.DataFrame(scaler.transform(test_df[numeric_cols].fillna(0)),
                            columns=[f"{c}_scaled" for c in numeric_cols], index=test_df.index)

train_final = pd.concat([train_df[["Price_INR","log_price"]], train_scaled], axis=1)
test_final = pd.concat([test_df[["Price_INR","log_price"]], test_scaled], axis=1)
train_final.shape, test_final.shape

((26069, 17), (6518, 17))

In [7]:
train_final.to_csv(f"{RAW}/train_preprocessed.csv", index=False)
test_final.to_csv(f"{RAW}/test_preprocessed.csv", index=False)
print(f"wrote train_preprocessed.csv ({len(train_final)} rows), test_preprocessed.csv ({len(test_final)} rows)")

wrote train_preprocessed.csv (26069 rows), test_preprocessed.csv (6518 rows)
